# 02 - Blanchard-Quah Long-Run Restrictions and Sign Restrictions

This notebook covers two advanced SVAR identification schemes:

1. **Blanchard-Quah (1989)**: Long-run restrictions
2. **Sign restrictions (Uhlig, 2005)**: Agnostic identification via inequality constraints

## Topics covered

- Long-run multipliers and the $C(1)$ matrix
- Identifying supply vs demand shocks via long-run neutrality
- Sign restrictions as an alternative to zero restrictions
- Comparing Cholesky, Blanchard-Quah, and sign restriction identification
- Median-target IRFs and credible bands

---

## From Short-Run to Long-Run Restrictions

In notebook 01, we used **short-run (contemporaneous)** restrictions to identify structural
shocks. But sometimes economic theory provides stronger predictions about **long-run** effects:

> "Demand shocks have no permanent effect on output" — Blanchard & Quah (1989)

The **long-run cumulative multiplier** is:

$$C(1) = \left(\sum_{i=0}^{\infty} \Phi_i\right) B_0^{-1} = \left(I - A_1 - A_2 - \cdots - A_p\right)^{-1} B_0^{-1}$$

The Blanchard-Quah restriction requires $C(1)$ to be **lower triangular**, meaning that the
second shock (demand) has **zero long-run effect** on the first variable (output).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox.models import VAR, SVAR

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_structural_irf, plot_irf_comparison
from data_generators import generate_blanchard_quah, generate_sign_restriction_dgp

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Blanchard-Quah Identification

### 1.1 The Setup

Blanchard & Quah (1989) studied a bivariate system:
- **Output growth** ($\Delta y_t$)
- **Unemployment** ($u_t$)

Two structural shocks:
- **Supply shock** ($\varepsilon_t^s$): has a **permanent** effect on the level of output
- **Demand shock** ($\varepsilon_t^d$): has only a **transitory** effect on output (zero long-run effect)

The key restriction: the cumulative effect of demand shocks on output growth sums to zero,
meaning the **long-run multiplier** of demand shocks on the output level is zero.

We first validate with synthetic data where we know the true DGP.

In [ ]:
# Generate synthetic Blanchard-Quah data
bq_data, bq_params = generate_blanchard_quah(n=500, seed=42)
bq_names = bq_params["variable_names"]

print(f"Variables: {bq_names}")
print(f"Data shape: {bq_data.shape}")
print(f"\nTrue structural impact matrix (B0_inv):")
print(pd.DataFrame(bq_params["B0_inv"].round(4), index=bq_names, columns=["supply", "demand"]))
print(f"\nTrue long-run matrix C(1) = (I-A1-A2)^{{-1}} @ B0_inv:")
print(pd.DataFrame(bq_params["long_run_matrix"].round(4), 
                    index=bq_names, columns=["supply", "demand"]))
print(f"\nNote: C(1)[0,1] = {bq_params['long_run_matrix'][0,1]:.6f} (should be ~0)")
print("=> Demand shock has zero long-run effect on output growth")

In [ ]:
# Fit VAR and apply Blanchard-Quah identification
var_bq = VAR(lags=2, trend="c")
var_bq_res = var_bq.fit(bq_data)

svar_bq = SVAR(var_bq_res, method="long_run")
results_bq = svar_bq.fit()

print(f"Identification method: {results_bq.method}")
print(f"\nEstimated structural impact matrix (B0_inv):")
print(pd.DataFrame(results_bq.A0_inv.round(4), index=bq_names, columns=["supply", "demand"]))

# Compute the estimated long-run matrix
I_minus_A = np.eye(2)
for lag_coef in results_bq.coefs:
    I_minus_A -= lag_coef
C1_est = np.linalg.inv(I_minus_A) @ results_bq.A0_inv

print(f"\nEstimated long-run matrix C(1):")
print(pd.DataFrame(C1_est.round(4), index=bq_names, columns=["supply", "demand"]))
print(f"\nC(1)[0,1] = {C1_est[0,1]:.6f} (should be ~0 by construction)")

In [ ]:
# Structural IRFs: Blanchard-Quah
irf_bq = results_bq.irf(periods=40)
print(f"IRF shape: {irf_bq.shape}")

fig = plot_structural_irf(
    irf_bq,
    variable_names=bq_names,
    shock_names=["Supply shock", "Demand shock"],
    title="Blanchard-Quah Structural IRF (Synthetic Data)",
    figsize=(10, 6),
)
plt.savefig(os.path.join("..", "outputs", "svar_bq_irf_synthetic.png"), bbox_inches="tight")
plt.show()

# Verify long-run restriction: cumulative IRF of demand shock on output → 0
cum_irf = np.cumsum(irf_bq, axis=0)
print(f"\nCumulative IRF of demand shock on output (last value): {cum_irf[-1, 0, 1]:.4f}")
print("(Should converge to ~0 for long-run neutrality)")

**Economic interpretation:**

- **Supply shock**: A positive supply shock permanently raises output. It reduces
  unemployment (the economy expands). The effect builds gradually and persists.

- **Demand shock**: A positive demand shock temporarily boosts output growth but the
  cumulative effect fades to zero — consistent with the long-run neutrality restriction.
  Unemployment falls temporarily but returns to its natural rate.

This decomposition separates the sources of business cycle fluctuations into
**permanent** (supply-driven) and **transitory** (demand-driven) components.

### 1.2 Blanchard-Quah on US Macro Data

Now let us apply the BQ decomposition to real data. We use GDP growth and unemployment
from the US macro dataset — the classic BQ specification.

In [ ]:
# Load US macro data
data_path = os.path.join("..", "data", "us_macro_quarterly.csv")
df = pd.read_csv(data_path, parse_dates=["date"])
df.set_index("date", inplace=True)

# BQ specification: [gdp_growth, unemployment]
bq_us_names = ["gdp", "unemployment"]
endog_bq_us = df[bq_us_names].values

print(f"BQ US data shape: {endog_bq_us.shape}")

# Fit VAR(4) and apply BQ identification
var_bq_us = VAR(lags=4, trend="c")
var_bq_us_res = var_bq_us.fit(endog_bq_us)

svar_bq_us = SVAR(var_bq_us_res, method="long_run")
results_bq_us = svar_bq_us.fit()

irf_bq_us = results_bq_us.irf(periods=24)

# Plot BQ IRFs on real data
fig = plot_structural_irf(
    irf_bq_us,
    variable_names=["GDP growth", "Unemployment"],
    shock_names=["Supply shock", "Demand shock"],
    title="Blanchard-Quah Decomposition (US Macro Data)",
    figsize=(10, 6),
)
plt.savefig(os.path.join("..", "outputs", "svar_bq_us_irf.png"), bbox_inches="tight")
plt.show()

## 2. Sign Restrictions (Uhlig, 2005)

Sign restrictions represent a **set-identification** approach. Instead of imposing exact
zero restrictions, we impose **inequality** constraints on the IRFs:

> "A monetary policy shock increases the interest rate and decreases output and inflation"

The algorithm:
1. Draw a random orthogonal matrix $Q$ (via QR decomposition of a random normal matrix)
2. Form a candidate $\tilde{B}_0^{-1} = P Q$ where $P$ is the Cholesky factor
3. Compute the candidate IRF
4. Check if the sign restrictions are satisfied
5. If yes, accept the draw; if no, discard and repeat

This produces a **set of admissible models**, not a single point estimate.
We report the **median IRF** and **credible bands** across accepted draws.

### 2.1 Sign Restrictions on Synthetic Data

We use a 3-variable system [output, inflation, rate] with known structural shocks:
- **Supply shock**: output (+), inflation (-), rate (-)
- **Demand shock**: output (+), inflation (+), rate (+)
- **Monetary policy shock**: output (-), inflation (-), rate (+)

In [ ]:
# Generate synthetic sign restriction data
sign_data, sign_params = generate_sign_restriction_dgp(n=500, seed=42)
sign_names = sign_params["variable_names"]
sign_shock_names = sign_params["shock_names"]

print(f"Variables: {sign_names}")
print(f"Structural shocks: {sign_shock_names}")
print(f"\nTrue sign pattern (impact matrix signs):")
print(pd.DataFrame(sign_params["sign_pattern"], 
                    index=sign_names, columns=sign_shock_names))
print("\n+1 = positive impact, -1 = negative impact")

# Fit reduced-form VAR
var_sign = VAR(lags=1, trend="c")
var_sign_res = var_sign.fit(sign_data)

In [ ]:
# Define sign restrictions
# Format: {(shock_idx, variable_idx, horizons): sign}
# We identify the monetary policy shock (shock 2):
#   - output decreases: (2, 0, range(0, 4)) = '-'
#   - inflation decreases: (2, 1, range(0, 4)) = '-'
#   - rate increases: (2, 2, range(0, 4)) = '+'

sign_restrictions = {
    # Monetary policy shock
    (2, 0, range(0, 4)): "-",   # output falls for 4 periods
    (2, 1, range(0, 4)): "-",   # inflation falls for 4 periods
    (2, 2, range(0, 4)): "+",   # rate rises for 4 periods
    # Supply shock  
    (0, 0, range(0, 4)): "+",   # output rises
    (0, 1, range(0, 4)): "-",   # inflation falls
    # Demand shock
    (1, 0, range(0, 4)): "+",   # output rises
    (1, 1, range(0, 4)): "+",   # inflation rises
}

print("Sign restrictions imposed:")
labels = {"+": "positive", "-": "negative"}
for (shock, var, horizons), sign in sign_restrictions.items():
    print(f"  Shock {sign_shock_names[shock]}: {sign_names[var]} must be "
          f"{labels[sign]} for horizons {horizons.start}-{horizons.stop-1}")

In [ ]:
# Fit SVAR with sign restrictions
svar_sign = SVAR(var_sign_res, method="sign", sign_restrictions=sign_restrictions)
results_sign = svar_sign.fit(n_draws=500, max_draws=50000)

print(f"Identification method: {results_sign.method}")
print(f"Accepted draws: {len(results_sign.accepted_draws)}")

# Get IRF with credible bands
median_irf, lower_irf, upper_irf = results_sign.irf_with_bands(periods=20, alpha=0.16)
print(f"\nMedian IRF shape: {median_irf.shape}")
print(f"68% credible bands computed")

In [ ]:
# Plot sign-restriction IRFs with credible bands
fig = plot_structural_irf(
    median_irf,
    variable_names=sign_names,
    shock_names=sign_shock_names,
    ci_lower=lower_irf,
    ci_upper=upper_irf,
    title="Sign-Restriction Identified IRFs (Median + 68% Bands)",
    figsize=(14, 9),
)
plt.savefig(os.path.join("..", "outputs", "svar_sign_irf_synthetic.png"), bbox_inches="tight")
plt.show()

### 2.2 Sign Restrictions on US Macro Data

Now let us apply sign restrictions to the full 4-variable US macro system to identify
a **monetary policy shock** without imposing a specific causal ordering.

In [ ]:
# Full US macro system with sign restrictions
us_var_names = ["gdp", "inflation", "fed_funds", "unemployment"]
endog_us = df[us_var_names].values

var_us = VAR(lags=4, trend="c")
var_us_res = var_us.fit(endog_us)

# Sign restrictions: identify monetary policy shock (shock index 2)
# Contractionary monetary policy:
#   - GDP falls (shock 2, var 0)
#   - Inflation falls (shock 2, var 1)
#   - Fed Funds rises (shock 2, var 2)
#   - Unemployment rises (shock 2, var 3)
sign_restr_us = {
    (2, 0, range(0, 4)): "-",   # GDP falls after monetary tightening
    (2, 1, range(0, 4)): "-",   # Inflation falls
    (2, 2, range(0, 4)): "+",   # Fed Funds rises
    (2, 3, range(0, 4)): "+",   # Unemployment rises
}

svar_sign_us = SVAR(var_us_res, method="sign", sign_restrictions=sign_restr_us)
results_sign_us = svar_sign_us.fit(n_draws=500, max_draws=50000)

median_us, lower_us, upper_us = results_sign_us.irf_with_bands(periods=20, alpha=0.16)

print(f"Accepted draws: {len(results_sign_us.accepted_draws)}")

# Plot monetary shock IRF with bands
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
horizons = np.arange(median_us.shape[0])
titles_us = ["GDP Growth", "Inflation", "Fed Funds Rate", "Unemployment"]

for i, (ax, title) in enumerate(zip(axes.flat, titles_us)):
    ax.plot(horizons, median_us[:, i, 2], "b-", linewidth=2, label="Median")
    ax.fill_between(horizons, lower_us[:, i, 2], upper_us[:, i, 2],
                    alpha=0.2, color="blue", label="68% band")
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_title(f"Response of {title} to Monetary Shock", fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle("Monetary Policy Shock (Sign Restrictions, US Data)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "svar_sign_us_monetary.png"), bbox_inches="tight")
plt.show()

## 3. Comparing Identification Schemes

Let us compare how different identification methods recover the monetary policy shock
on the same US macro dataset. This highlights the trade-offs between approaches.

In [ ]:
# Cholesky identification on same data
svar_chol_us = SVAR(var_us_res, method="cholesky")
results_chol_us = svar_chol_us.fit()
irf_chol_us = results_chol_us.irf(periods=20)

# Compare Cholesky vs Sign restrictions for the monetary shock
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (ax, title) in enumerate(zip(axes.flat, titles_us)):
    # Cholesky
    ax.plot(horizons, irf_chol_us[:, i, 2], "r-", linewidth=1.8, label="Cholesky")
    # Sign restrictions (median + bands)
    ax.plot(horizons, median_us[:, i, 2], "b-", linewidth=1.8, label="Sign (median)")
    ax.fill_between(horizons, lower_us[:, i, 2], upper_us[:, i, 2],
                    alpha=0.15, color="blue")
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_title(f"{title}", fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle("Monetary Shock: Cholesky vs Sign Restrictions (US Data)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "svar_comparison_chol_sign.png"), bbox_inches="tight")
plt.show()

**Comparison summary:**

| Method | Type | Restrictions | Pros | Cons |
|--------|------|-------------|------|------|
| **Cholesky** | Point-identified | $K(K-1)/2$ zero restrictions (contemporaneous) | Simple, unique solution | Depends on ordering; strong assumptions |
| **Blanchard-Quah** | Point-identified | $K(K-1)/2$ zero restrictions (long-run) | Economically motivated; no ordering needed | Requires stationarity; sensitive to lag length |
| **Sign restrictions** | Set-identified | Inequality constraints on IRFs | Minimal assumptions; robust | Wide bands; multiple models consistent |

**Key insight:** Sign restrictions avoid the "price puzzle" (inflation rising after
monetary tightening) by construction — we impose that inflation must fall. The Cholesky
approach may show the puzzle depending on the ordering and sample period.

In [ ]:
# FEVD comparison: Cholesky vs BQ on the bivariate system
# BQ FEVD
fevd_bq_us = results_bq_us.fevd(periods=20)

# Cholesky on same bivariate system
svar_chol_bv = SVAR(var_bq_us_res, method="cholesky")
results_chol_bv = svar_chol_bv.fit()
fevd_chol_bv = results_chol_bv.fevd(periods=20)

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
h = np.arange(fevd_bq_us.shape[0])
bq_shock = ["Supply", "Demand"]

for row, method_name, fevd_data in [(0, "Blanchard-Quah", fevd_bq_us), 
                                      (1, "Cholesky", fevd_chol_bv)]:
    for col, vname in enumerate(["GDP growth", "Unemployment"]):
        ax = axes[row, col]
        bottom = np.zeros_like(h, dtype=float)
        for j, sname in enumerate(bq_shock):
            ax.fill_between(h, bottom, bottom + fevd_data[:, col, j],
                           alpha=0.7, label=sname)
            bottom += fevd_data[:, col, j]
        ax.set_title(f"{method_name}: {vname}")
        ax.set_ylim(0, 1)
        ax.set_xlabel("Horizon")
        ax.grid(True, alpha=0.2)
        if col == 0 and row == 0:
            ax.legend(fontsize=8)

plt.suptitle("FEVD: Blanchard-Quah vs Cholesky (Bivariate System)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "svar_fevd_bq_vs_cholesky.png"), bbox_inches="tight")
plt.show()

## Exercicio

### Exercicio 1: Blanchard-Quah com mais lags

A decomposicao BQ pode ser sensivel ao numero de lags do VAR. Reestime o modelo
BQ com o dataset US macro usando VAR(2) e VAR(8). Compare as IRFs.

1. A IRF cumulativa do choque de demanda converge para zero em todos os casos?
2. Como o numero de lags afeta a magnitude do choque de oferta no longo prazo?
3. Use o criterio AIC para escolher o lag otimo e reestime.

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

# Dica:
# for p in [2, 4, 8]:
#     var_tmp = VAR(lags=p, trend="c")
#     res_tmp = var_tmp.fit(endog_bq_us)
#     svar_tmp = SVAR(res_tmp, method="long_run")
#     results_tmp = svar_tmp.fit()
#     irf_tmp = results_tmp.irf(periods=40)
#     cum_irf = np.cumsum(irf_tmp, axis=0)
#     print(f"Lags={p}: cum IRF demand→output at h=40: {cum_irf[-1, 0, 1]:.4f}")

### Exercicio 2: Sign restrictions com restricoes alternativas

Reidentifique o choque monetario no dataset US macro usando apenas 2 restricoes
(em vez de 4):
- Fed Funds sobe: `(2, 2, range(0, 2)): "+"`
- GDP cai: `(2, 0, range(0, 2)): "-"`

Compare com o resultado usando 4 restricoes. O que acontece com a largura das
bandas de credibilidade? Restricoes mais fracas levam a mais ou menos incerteza?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

# Dica:
# sign_restr_weak = {
#     (2, 2, range(0, 2)): "+",
#     (2, 0, range(0, 2)): "-",
# }
# svar_weak = SVAR(var_us_res, method="sign", sign_restrictions=sign_restr_weak)
# results_weak = svar_weak.fit(n_draws=500, max_draws=50000)
# median_w, lower_w, upper_w = results_weak.irf_with_bands(periods=20, alpha=0.16)

---

## Resumo

Neste notebook aprendemos:

1. **Blanchard-Quah (1989)**: usa restricoes de longo prazo para separar choques
   permanentes (oferta) de transitorios (demanda). A matriz $C(1)$ deve ser triangular
   inferior. Nao requer ordenamento das variaveis.

2. **Sign restrictions (Uhlig, 2005)**: impoe apenas desigualdades nas IRFs,
   produzindo um **conjunto** de modelos admissiveis (set identification).
   Mais agnostica, mas com bandas mais largas.

3. **Relacao entre forma reduzida e estrutural**: $u_t = B_0^{-1}\varepsilon_t$.
   Cada metodo de identificacao impoe restricoes diferentes sobre $B_0^{-1}$
   para recuperar os choques estruturais $\varepsilon_t$ a partir dos residuos
   reduzidos $u_t$.

4. **Trade-off fundamental**: restricoes mais fortes (Cholesky, BQ) dao respostas
   pontuais mas dependem fortemente das hipoteses. Restricoes mais fracas (sign)
   sao mais robustas mas menos informativas.

No proximo notebook, exploraremos **VAR Bayesiano (BVAR)** com prior Minnesota.